# 01 — Exploratory Data Analysis

Explore the Garbage Classification V2 dataset: class distribution, sample images, image dimensions, and color analysis.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

from src.data_loader import discover_class_names

sns.set_style("whitegrid")
DATA_DIR = os.path.join("..", "data", "raw", "standardized_256")

## 1. Discover Classes and Count Samples

In [ ]:
class_names = discover_class_names(DATA_DIR)
print(f"Found {len(class_names)} classes: {class_names}")

class_counts = {}
for cls in class_names:
    cls_dir = os.path.join(DATA_DIR, cls)
    count = len([f for f in os.listdir(cls_dir) if os.path.isfile(os.path.join(cls_dir, f))])
    class_counts[cls] = count

print(f"\nTotal images: {sum(class_counts.values())}")
for cls, cnt in class_counts.items():
    print(f"  {cls}: {cnt}")

## 2. Class Distribution Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = sns.color_palette("viridis", len(class_names))
bars = ax.bar(class_counts.keys(), class_counts.values(), color=colors)
ax.set_xlabel("Class")
ax.set_ylabel("Number of Images")
ax.set_title("Class Distribution")
plt.xticks(rotation=45, ha="right")

for bar, cnt in zip(bars, class_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(cnt), ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Sample Images Grid (5 per class)

In [ ]:
n_samples = 5
fig, axes = plt.subplots(len(class_names), n_samples, figsize=(3 * n_samples, 3 * len(class_names)))

for row, cls in enumerate(class_names):
    cls_dir = os.path.join(DATA_DIR, cls)
    files = sorted(os.listdir(cls_dir))[:n_samples]
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(cls_dir, fname)).convert("RGB")
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(cls, fontsize=10, fontweight="bold")

plt.suptitle("Sample Images per Class", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Image Dimension Analysis

In [ ]:
widths, heights = [], []
sample_limit = 200  # sample per class to keep it fast

for cls in class_names:
    cls_dir = os.path.join(DATA_DIR, cls)
    files = os.listdir(cls_dir)[:sample_limit]
    for fname in files:
        with Image.open(os.path.join(cls_dir, fname)) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)

print(f"Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(widths, bins=30, color="steelblue", edgecolor="white")
ax1.set_title("Width Distribution")
ax1.set_xlabel("Pixels")
ax2.hist(heights, bins=30, color="coral", edgecolor="white")
ax2.set_title("Height Distribution")
ax2.set_xlabel("Pixels")
plt.tight_layout()
plt.show()

## 5. Per-Class Mean Pixel Intensity

In [ ]:
class_means = {}
for cls in class_names:
    cls_dir = os.path.join(DATA_DIR, cls)
    files = os.listdir(cls_dir)[:50]
    means = []
    for fname in files:
        img = np.array(Image.open(os.path.join(cls_dir, fname)).convert("RGB"))
        means.append(img.mean())
    class_means[cls] = np.mean(means)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(class_means.keys(), class_means.values(), color=sns.color_palette("coolwarm", len(class_names)))
ax.set_ylabel("Mean Pixel Intensity")
ax.set_title("Per-Class Mean Pixel Intensity")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Color Histogram Comparison

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, cls in enumerate(class_names[:9]):
    cls_dir = os.path.join(DATA_DIR, cls)
    files = os.listdir(cls_dir)[:30]
    all_r, all_g, all_b = [], [], []
    for fname in files:
        img = np.array(Image.open(os.path.join(cls_dir, fname)).convert("RGB"))
        all_r.extend(img[:,:,0].flatten()[::10])  # subsample for speed
        all_g.extend(img[:,:,1].flatten()[::10])
        all_b.extend(img[:,:,2].flatten()[::10])

    axes[i].hist(all_r, bins=50, alpha=0.5, color="red", label="R")
    axes[i].hist(all_g, bins=50, alpha=0.5, color="green", label="G")
    axes[i].hist(all_b, bins=50, alpha=0.5, color="blue", label="B")
    axes[i].set_title(cls, fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle("RGB Color Histograms per Class", fontsize=14)
plt.tight_layout()
plt.show()

## Observations

- **Class imbalance:** Some classes may have significantly more images than others — we'll use class weights during training.
- **Image dimensions:** Most images are 512x384 or similar; all will be resized to 224x224 for model input.
- **Color variation:** Each class shows distinctive color distributions (e.g., vegetation is greener, metal tends to be more uniform). This suggests color features may help classification.
- **Visual similarity:** Some classes (e.g., Paper vs. Cardboard, Plastic vs. Glass) may be challenging to distinguish — transfer learning should help.